# All Convolutional Net


In [4]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics

class ComprehensiveConvNet(pl.LightningModule):
    def __init__(self, classes_count=10):
        super(ComprehensiveConvNet, self).__init__()
        # Initializing convolutional layers with detailed specifications
        self.conv_layer1 = nn.Conv2d(1, 96, kernel_size=3, padding=1)
        self.conv_layer2 = nn.Conv2d(96, 96, kernel_size=3, padding=1)
        self.conv_layer3 = nn.Conv2d(96, 96, kernel_size=3, stride=2, padding=1)  # Adding stride for downsampling
        self.conv_layer4 = nn.Conv2d(96, 192, kernel_size=3, padding=1)
        self.conv_layer5 = nn.Conv2d(192, 192, kernel_size=3, padding=1)
        self.conv_layer6 = nn.Conv2d(192, 192, kernel_size=3, stride=2, padding=1)  # Another layer of downsampling
        self.conv_layer7 = nn.Conv2d(192, 192, kernel_size=3, padding=1)
        self.conv_layer8 = nn.Conv2d(192, 192, kernel_size=1)  # Using 1x1 convolutions
        self.conv_layer9 = nn.Conv2d(192, classes_count, kernel_size=1)  # Last 1x1 convolution for class predictions

        self.avg_pool = nn.AdaptiveAvgPool2d(1)  # Global Average Pooling
        self.metric_accuracy = torchmetrics.Accuracy(classes_count=classes_count, average="macro", task="multiclass")

    def forward(self, inputs):
        inputs = F.relu(self.conv_layer1(inputs))
        inputs = F.relu(self.conv_layer2(inputs))
        inputs = F.relu(self.conv_layer3(inputs))
        inputs = F.relu(self.conv_layer4(inputs))
        inputs = F.relu(self.conv_layer5(inputs))
        inputs = F.relu(self.conv_layer6(inputs))
        inputs = F.relu(self.conv_layer7(inputs))
        inputs = F.relu(self.conv_layer8(inputs))
        inputs = self.conv_layer9(inputs)
        inputs = self.avg_pool(inputs)
        inputs = inputs.view(inputs.size(0), -1)  # Reshape before sending to output
        return inputs

    def on_train_batch(self, batch, batch_index):
        inputs, targets = batch
        predictions = self(inputs)
        train_loss = F.cross_entropy(predictions, targets)
        self.log("train_loss", train_loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return train_loss

    def on_validate_batch(self, batch, batch_index):
        inputs, targets = batch
        predictions = self(inputs)
        validation_loss = F.cross_entropy(predictions, targets)
        self.metric_accuracy(predictions, targets)
        self.log("validation_loss", validation_loss, on_epoch=True, prog_bar=True)
        self.log("validation_accuracy", self.metric_accuracy, on_epoch=True, prog_bar=True)

    def on_test_batch(self, batch, batch_index):
        inputs, targets = batch
        predictions = self(inputs)
        test_loss = F.cross_entropy(predictions, targets)

        self.metric_accuracy(predictions, targets)

        self.log("test_accuracy", self.metric_accuracy)
        self.log("test_loss", test_loss)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        return optimizer


The Imagenette dataset is a smaller subset of 10 easily classified classes from Imagenet. It is available to download from `torchvision`, as shown in the cell below. There are 3 different sizes of the images available. Feel free to use whichever version you prefer. It might make a difference in the performance of your model.

**Note: After downloading the Imagenette dataset, you will need to set `download=False` in the cell below to avoid errors.**

In [5]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from torchvision import transforms, datasets
import torch
from torch.utils.data import DataLoader, random_split

# Data preprocessing setups for image transformation
initial_transforms = transforms.Compose([
    transforms.CenterCrop(160),
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)),
    transforms.Grayscale()  # Ensuring images are grayscale to reduce complexity
])

evaluation_transforms = transforms.Compose([
    transforms.CenterCrop(160),
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)),
    transforms.Grayscale()  # Consistent transformation for evaluation phase
])

# Load and organize the training data
imagenette_training = datasets.Imagenette("../data/imagenette/train/", split="train", size="160px", download=False, transform=initial_transforms)

# Calculate the size for training and validation splits
training_size = int(0.9 * len(imagenette_training))
validation_size = len(imagenette_training) - training_size

# Seed for reproducibility when splitting datasets
split_seed = torch.Generator().manual_seed(42)
training_data, validation_data = random_split(imagenette_training, [training_size, validation_size], generator=split_seed)
validation_data.dataset.transform = evaluation_transforms

# DataLoader configurations for training and validation sets
training_loader = DataLoader(
    training_data, batch_size=128, num_workers=8, shuffle=True, persistent_workers=True
)
validation_loader = DataLoader(
    validation_data, batch_size=128, num_workers=8, shuffle=False, persistent_workers=True
)

# Setup for the test dataset with similar transformations
test_data = datasets.Imagenette("../data/imagenette/test/", split="val", size="160px", download=False, transform=evaluation_transforms)

# Instantiate the CNN model
convolutional_network = AllConvNet()

# Setup EarlyStopping to prevent overfitting
early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=5  # Number of epochs to wait after min has been hit. After this number of epochs stop training.
)

# Configuration for saving model checkpoints based on validation loss
model_saver = ModelCheckpoint(
    monitor="val_loss",
    mode="min"  # Save the model when the min validation loss is achieved
)


In [6]:
# Fit the model
torch.set_float32_matmul_precision("high")
trainer = L.Trainer(callbacks=[early_stop_callback, checkpoint_callback], max_epochs=-1)

trainer.fit(model=model, train_dataloaders=train_loader, val_dataloaders=val_loader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /Users/kshitij/Desktop/CSE/CSE4310ComputerVision/CSE4310-CNN_Image_Classification-main/CNN_notebooks/lightning_logs/version_0/checkpoints exists and is not empty.

   | Name            | Type               | Params | Mode 
----------------------------------------------------------------
0  | conv1           | Conv2d             | 960    | train
1  | conv2           | Conv2d             | 83.0 K | train
2  | conv3           | Conv2d             | 83.0 K | train
3  | conv4           | Conv2d             | 166 K  | train
4  | conv5           | Conv2d             | 331 K  | train
5  | conv6           | Conv2d             | 331 K  | train
6  | conv7           | Conv2d             | 331 K  | train
7  | conv8           | Conv2d 

Sanity Checking: |                                        | 0/? [00:00<?, ?it/s]

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/__init__.py", line 1755, in <module>

Detected KeyboardInterrupt, attempting graceful shutdown ...
    from .functional import *  # noqa: F403
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/functional.py", line 10, in <module>
    import torch.nn.functional as F
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/nn/__init__.py", line 2, in <module>
    from .modules import *  # noqa: F403
  File "/Library/

NameError: name 'exit' is not defined

In [ ]:
# Evaluate the model on the test set
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=256, num_workers=8, shuffle=False, persistent_workers=True
)
trainer.test(model=model, dataloaders=test_loader)